### 0. Import modules

In [7]:
import stable_baselines3 as sb3
import gym_unbalanced_disk
import gymnasium as gym
import numpy as np


### 1 Get the saving dirs, Prepare the Env, prepare the callback funcitons

In [ ]:
"""
Train an A2C/PPO agent on the unbalanced-disk swing-up task using stable-baselines3.

Run from anywhere (the gym_unbalanced_disk import registers the env id):
    python a2c.py

View the training logs with:
    tensorboard --logdir ./tensorboard_logs
"""

import time
from pathlib import Path

import numpy as np
import gymnasium as gym

# Importing gym_unbalanced_disk registers the 'unbalanced-disk-v0' env id with
# gymnasium, so it must be imported before any gym.make(...) call.
import gym_unbalanced_disk  # noqa: F401  (imported for its registration side effect)
import stable_baselines3 as sb3
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import (
    EvalCallback,
    CallbackList,
    BaseCallback,
)


def get_save_dirs(model_name: str):
    # All artifacts (checkpoints, logs) are written next to this script, regardless
    # of the directory the script is launched from.
    try:
        HERE = Path(__file__).resolve().parent
    except NameError:
        # __file__ is undefined in a Jupyter notebook; use the working dir
        HERE = Path.cwd()
    TB_LOG_DIR = HERE / "tensorboard_logs"
    BEST_MODEL_DIR = HERE / f"{model_name}_best"
    EVAL_LOG_DIR = HERE / f"{model_name}_eval_logs"
    CHECKPOINT_DIR = HERE / f"{model_name}_checkpoints"
    FINAL_MODEL_PATH = HERE / f"{model_name}_unbalanced_disk"
    return TB_LOG_DIR, BEST_MODEL_DIR, EVAL_LOG_DIR, CHECKPOINT_DIR, FINAL_MODEL_PATH


# ---------------------------------------------------------------------------
# Environment factory
# ---------------------------------------------------------------------------
def make_env():
    """
    Build a single training/eval environment.
    """
    env = gym.make("unbalanced-disk-v0", dt=0.025, umax=3.0)
    env = Monitor(env)
    return env


# ---------------------------------------------------------------------------
# Standalone evaluation (used for a final report after training)
# ---------------------------------------------------------------------------
def evaluate(model, env, num_episodes=10):
    """
    Run the (deterministic) policy for a number of episodes and return the
    mean total reward per episode.
    """
    total_rewards = []
    for _ in range(num_episodes):
        obs, info = env.reset()
        done = False
        episode_reward = 0.0
        while not done:
            action, _states = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
            done = terminated or truncated
        total_rewards.append(episode_reward)
    return float(np.mean(total_rewards))

def visualize_trained_policy(model_path: Path, deterministic: bool = False):
# -----------------------------------------------------------------------
# Visualise the trained policy
# -----------------------------------------------------------------------
# A human-rendered env to watch the swing-up. render_mode="human" opens a
# pygame window.
    if "ppo" in model_path.name:
        model = sb3.PPO.load(str(model_path), device="cpu")
    else:
        model = sb3.A2C.load(str(model_path), device="cpu")
    vis_env = gym.make("unbalanced-disk-v0", dt=0.025, umax=3.0, render_mode="human")
    obs, info = vis_env.reset()
    try:
        for _ in range(200):
            action, _states = model.predict(obs, deterministic=deterministic)
            obs, reward, terminated, truncated, info = vis_env.step(action)
            print(obs, reward)
            vis_env.render()
            time.sleep(1 / 24)
            if terminated or truncated:
                obs, info = vis_env.reset()   # unpack the (obs, info) tuple
    finally:  # always run, even on Ctrl-C, so the window/resources are released
        vis_env.close()


class RenderEvalCallback(BaseCallback):
    """Every `render_freq` steps, play one deterministic episode in a
    human-rendered env so the current policy can be watched. Kept separate
    from EvalCallback so metric logging (eval_freq) stays untouched."""
    def __init__(self, render_freq=50000, n_episodes=1, max_steps=200, verbose=0):
        super().__init__(verbose)
        self.render_freq = render_freq
        self.n_episodes = n_episodes
        self.max_steps = max_steps

    def _on_step(self) -> bool:
        # n_calls increments once per _on_step; with a single env this == timesteps.
        # For multiple envs use self.num_timesteps instead.
        if self.n_calls % self.render_freq == 0:
            vis_env = gym.make("unbalanced-disk-v0", dt=0.025, umax=3.0,
                               render_mode="human")
            try:
                for _ in range(self.n_episodes):
                    obs, info = vis_env.reset()
                    for _ in range(self.max_steps):
                        action, _ = self.model.predict(obs, deterministic=True)
                        obs, reward, terminated, truncated, info = vis_env.step(action)
                        vis_env.render()
                        time.sleep(1 / 24)
                        if terminated or truncated:
                            break
            finally:
                vis_env.close()
        return True


def get_callbacks(eval_env):


    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(BEST_MODEL_DIR),
        log_path=str(EVAL_LOG_DIR),
        eval_freq=25000,       # run an evaluation every 10k training steps
        n_eval_episodes=5,
        deterministic=True,
        render=False,
    )

    # Metric eval every 20k steps + a watchable rendered rollout every 50k steps.
    callbacks = CallbackList([eval_callback, RenderEvalCallback(render_freq=50000)])

    return callbacks

In [ ]:

# Separate environments for training and for periodic evaluation, so eval
# episodes do not interfere with the training rollouts.
train_env = make_env()
eval_env = make_env()
TB_LOG_DIR, BEST_MODEL_DIR, EVAL_LOG_DIR, CHECKPOINT_DIR, FINAL_MODEL_PATH = get_save_dirs("a2c")

# A2C with the default MLP policy. device="cpu" is correct here: the network
# is tiny and the bottleneck is the scipy ODE solve inside each env step,
# so a GPU would not help.
model = sb3.A2C(
    "MlpPolicy",
    train_env,
    verbose=0,
    device="cpu",
    tensorboard_log=str(TB_LOG_DIR),
)

callbacks = get_callbacks(eval_env)

# Train. progress_bar=True needs the 'tqdm' and 'rich' packages
# (set it to False if they are not installed).
model.learn(
    total_timesteps=50000,
    callback=callbacks,
    tb_log_name="a2c",
    progress_bar=True,
)

# Save the final (last) model. The best model seen during training was
# already saved by EvalCallback at BEST_MODEL_DIR / "best_model.zip".
model.save(str(FINAL_MODEL_PATH))
print(f"Final model saved to: {FINAL_MODEL_PATH}.zip")
print(f"Best model saved to : {BEST_MODEL_DIR / 'best_model.zip'}")

# Prefer the best checkpoint for evaluation/visualisation; fall back to the
# final model if (e.g. for a very short run) no eval ever triggered.
best_model_path = BEST_MODEL_DIR / "best_model.zip"
if best_model_path.exists():
    model = sb3.A2C.load(str(best_model_path), device="cpu")
    print("Loaded best model for evaluation.")

# Final evaluation report on a fresh eval env.
mean_reward = evaluate(model, eval_env, num_episodes=10)
print(f"\nFinal mean reward over 10 episodes: {mean_reward:.2f}")

train_env.close()
eval_env.close()

Eval num_timesteps=4000, episode_reward=24.54 +/- 0.01

Episode length: 200.00 +/- 0.00

New best mean reward!

Eval num_timesteps=8000, episode_reward=26.75 +/- 0.02

Episode length: 200.00 +/- 0.00

New best mean reward!

KeyboardInterrupt: 

In [ ]:
TB_LOG_DIR, BEST_MODEL_DIR, EVAL_LOG_DIR, CHECKPOINT_DIR, FINAL_MODEL_PATH = get_save_dirs("ppo")
train_env = make_env()
eval_env = make_env()

model = sb3.PPO(
    "MlpPolicy",
    train_env,
    verbose=0,
    device="cpu",
    tensorboard_log=str(TB_LOG_DIR),
)


callbacks = get_callbacks(eval_env)
model.learn(
        total_timesteps=1000000,
        callback=callbacks,
        tb_log_name="ppo",
        progress_bar=True,
    )
# Save the final (last) model. The best model seen during training was
# already saved by EvalCallback at BEST_MODEL_DIR / "best_model.zip".
model.save(str(FINAL_MODEL_PATH))
print(f"Final model saved to: {FINAL_MODEL_PATH}.zip")
print(f"Best model saved to : {BEST_MODEL_DIR / 'best_model.zip'}")

# Prefer the best checkpoint for evaluation/visualisation; fall back to the
# final model if (e.g. for a very short run) no eval ever triggered.
best_model_path = BEST_MODEL_DIR / "best_model.zip"
if best_model_path.exists():
    model = sb3.PPO.load(str(best_model_path), device="cpu")
    print("Loaded best model for evaluation.")

# Final evaluation report on a fresh eval env.
mean_reward = evaluate(model, eval_env, num_episodes=10)
print(f"\nFinal mean reward over 10 episodes: {mean_reward:.2f}")


train_env.close()
eval_env.close()

visualize_trained_policy(best_model_path)


Output()

Eval num_timesteps=10000, episode_reward=0.00 +/- 0.00

Episode length: 200.00 +/- 0.00

New best mean reward!

Eval num_timesteps=20000, episode_reward=0.00 +/- 0.00

...

Episode length: 200.00 +/- 0.00

Eval num_timesteps=250000, episode_reward=26.64 +/- 0.02

Episode length: 200.00 +/- 0.00

Eval num_timesteps=260000, episode_reward=26.63 +/- 0.02

Episode length: 200.00 +/- 0.00

In [6]:
# --- Continue training the PPO model for another 1M steps ---
# SB3 continues from the existing weights; reset_num_timesteps=False keeps the
# global step counter going so TensorBoard shows one continuous curve instead
# of restarting from step 0.
#
# If the kernel was restarted and `model` is gone, uncomment the load line:
# model = sb3.PPO.load(str(FINAL_MODEL_PATH), env=train_env, device="cpu")
TB_LOG_DIR, BEST_MODEL_DIR, EVAL_LOG_DIR, CHECKPOINT_DIR, FINAL_MODEL_PATH = get_save_dirs("ppo")
train_env = make_env()
eval_env = make_env()
callbacks = get_callbacks(eval_env)


best_model_path = BEST_MODEL_DIR / "best_model.zip"
if best_model_path.exists():
    model = sb3.PPO.load(str(best_model_path), device="cpu")
    print("Loaded best model for evaluation.")
model.set_env(train_env)   # make sure an env is attached after a reload

model.learn(
    total_timesteps=1_000_000,
    callback=callbacks,
    tb_log_name="ppo",
    reset_num_timesteps=False,   # <-- continue, don't restart
    progress_bar=True,
)

model.save(str(FINAL_MODEL_PATH))
print(f"Continued model saved to: {FINAL_MODEL_PATH}.zip")
print(f"Best model still at     : {BEST_MODEL_DIR / 'best_model.zip'}")


Output()

Loaded best model for evaluation.


Eval num_timesteps=830000, episode_reward=216.75 +/- 0.20

Episode length: 200.00 +/- 0.00

New best mean reward!

...

Eval num_timesteps=1810000, episode_reward=9872.25 +/- 0.63

Episode length: 200.00 +/- 0.00

Eval num_timesteps=1820000, episode_reward=9875.14 +/- 0.53

Episode length: 200.00 +/- 0.00

Continued model saved to: /home/nemo/Documents/GitHub/IML/DesignProject/DesignProject/Part2_Control/A2C/ppo_unbalanced_disk.zip
Best model still at     : /home/nemo/Documents/GitHub/IML/DesignProject/DesignProject/Part2_Control/A2C/ppo_best/best_model.zip


In [7]:
visualize_trained_policy(best_model_path)


[0.02413119 1.8997242 ] 0.07236624900973464


/home/nemo/Documents/GitHub/IML/DesignProject/.venv/lib/python3.12/site-packages/gymnasium/utils/passive_env_checker.py:265: UserWarning: WARN: Human rendering should return `None`, got <class 'bool'>
  logger.warn(


[0.09231649 3.586048  ] 0.259585402944853
[0.2017717 4.939835 ] 0.4980222564728516
[0.3373737 5.868454 ] 0.7168213722121872
[0.49202442 6.3308625 ] 0.8610709330871051
[0.65095866 6.3449264 ] 0.9070969041868302
[0.8044778 5.9606605] 0.8637139291430658
[0.943853  5.2626157] 0.7611104517357796
[1.0670372 4.3421903] 0.6354577454032573
[1.1591434 3.286286 ] 0.5172206318735962
[1.2274508 2.1609328] 0.42625640822713134
[1.2698362 1.0166903] 0.37222423371196905
[ 1.2772473 -0.2846505] 0.35708415701452745
[ 1.2478129 -2.2458506] 0.44103124600704835
[ 1.1295238 -7.0374346] 1.2771576545973016
[  0.89698434 -11.424908  ] 2.798845620368409
[  0.56456375 -15.091746  ] 4.633213882376156
[  0.15451485 -17.63654   ] 6.226535719018571
[ -0.3073388 -18.747456 ] 7.051924224827547
[ -0.7725969 -18.443514 ] 6.94521548001362
[ -1.2193276 -17.124289 ] 6.193589547385961
[ -1.626637 -15.379932] 5.260143396030601
[ -1.9911506 -13.74991  ] 4.485576101810887
[ -2.3174138 -12.606408 ] 4.017766055251952
[ -2.6263933